# Use case: one recommendation

A minimal example: given **one** pantry, get the Top-5 recipes. It uses the already-trained
final model `models/lda_model.pkl` (K auto-selected) — **no retraining**.

## 1 · Load the final model

The first cell points the working dir at `model/` and puts `src/` on the path.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":     # launched from model/notebooks/
    os.chdir("..")                                   # -> model/
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import pandas as pd
import recipe_recommender as rr

df = pd.read_csv("../data/recipes_clean.csv")
model = rr.load_model("models/lda_model.pkl")    # the final model (K auto-selected)
rr._STATE = {"model": model, "df_id": id(df)}    # let recommend() reuse it (no retrain)

print(f"Loaded final model (K={model.best_k}) and {len(df):,} recipes")

Loaded final model (K=4) and 53,573 recipes


## 2 · My pantry tonight

Tonight I have these on hand:

In [2]:
pantry = ["chicken", "garlic", "onion", "olive oil", "tomato",
          "rice", "salt", "black pepper"]

recs = rr.recommend(pantry, df, top_n=5)         # one call -> Top-5
pd.DataFrame(recs)[["recipe_name", "coverage", "missing_ingredients",
                    "predicted_rating", "posterior_uncertainty", "score"]]

[filter_candidates] threshold=0.7: 25 candidate recipes


,recipe_name,coverage,missing_ingredients,predicted_rating,posterior_uncertainty,score
0,solo sweet onion rice,4/5 ingredients,[chicken stock],4.088,0.2040,1.4928
1,rice for dummies and in the microwave too,3/4 ingredients,[water],4.385,0.0691,1.2712
2,roasted cauliflower with garlic,4/5 ingredients,[cauliflower],4.638,0.4473,1.0670
3,shmaltz,3/3 ingredients,[],4.724,0.2649,1.0438
4,stewed tomatoes and garbanzo beans,5/7 ingredients,"[garlic clove, garbanzo bean]",4.543,0.4028,0.4143


## 3 · Reading the top pick

- **coverage** — how many of the recipe's ingredients you already have.
- **missing_ingredients** — what you'd still need to buy.
- **flavor_tags** — the two flavor topics from the posterior.
- **predicted_rating** — Bayesian-shrunk rating.
- **posterior_uncertainty** — how (un)sure the model is (std of flavor alignment across Bootstrap draws).

In [3]:
top = recs[0]
print("Top pick:", top["recipe_name"])
print("  coverage         :", top["coverage"])
print("  missing          :", top["missing_ingredients"] or "nothing - you can make it now!")
print("  flavor tags      :", "  |  ".join(top["flavor_tags"]))
print("  predicted rating :", top["predicted_rating"], "/ 5")
print("  uncertainty (std):", top["posterior_uncertainty"])
print("  score            :", top["score"])

Top pick: solo sweet onion rice
  coverage         : 4/5 ingredients
  missing          : ['chicken stock']
  flavor tags      : salt / onion / water  |  cheddar cheese / onion / green onion
  predicted rating : 4.088 / 5
  uncertainty (std): 0.204
  score            : 1.4928


## 4 · Add a constraint

Same single call, just with options: **vegetarian only**, and it **must use the tomato**.

In [4]:
veg = rr.recommend(pantry, df, diet="vegetarian", must_use=["tomato"], top_n=5)
pd.DataFrame(veg)[["recipe_name", "coverage", "missing_ingredients", "predicted_rating"]]

[filter_candidates] threshold=0.7: 25 candidate recipes
[filters] diet=vegetarian, must_use=['tomato']: 25 -> 8 recipes


,recipe_name,coverage,missing_ingredients,predicted_rating
0,stewed tomatoes and garbanzo beans,5/7 ingredients,"[garlic clove, garbanzo bean]",4.543
1,chevy s salsa original recipe,5/7 ingredients,"[jalapeno pepper, cilantro]",3.697
2,simple roasted tomato and garlic sauce,3/3 ingredients,[],3.840
3,chef flower s simple avocado dip,5/7 ingredients,"[avocado, lemon juice]",4.817
4,authentic italian spaghetti sauce,5/7 ingredients,"[basil, crushed red pepper flake]",4.345


## That's it

One import, one `recommend()` call. For the other options (`top_n` / `exclude` /
`must_use` / `diet` / `diversity`) and field meanings, see Step 5 in the README.